# FA-UNIFEWS: Complete Experiment Suite (Google Colab)

**Instructions:**
1. Upload the `Unifews/` folder to your Google Drive under `MyDrive/Research/Unifews/`
2. Set Runtime → Change runtime type → **GPU (T4)**
3. Run all cells top-to-bottom (Ctrl+F9)

**Estimated time on T4 GPU: ~15-20 min for all single-seed experiments**

## Experiments
| # | Experiment | Runs | Est. time |
|---|-----------|------:|----------:|
| 1 | Main comparison (3 methods × 6 datasets) | 18 | ~5 min |
| 2 | MLP baselines | 6 | ~2 min |
| 3 | Dense GCN baselines | 6 | ~3 min |
| 4 | Ablation study | 14 | ~4 min |
| 5 | Sparsity sweep | 48 | ~10 min |
| 6 | Edge homophily | - | instant |
| **Total** | | **92** | **~25 min** |

## 0. Setup: Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# ====== CHANGE THIS PATH if your Unifews folder is elsewhere ======
DRIVE_UNIFEWS = '/content/drive/MyDrive/Research/Unifews'
# ==================================================================

# Auto-detect if code is nested (Unifews/Unifews/run_fb.py vs Unifews/run_fb.py)
if os.path.exists(os.path.join(DRIVE_UNIFEWS, 'run_fb.py')):
    DRIVE_CODE_ROOT = DRIVE_UNIFEWS
elif os.path.exists(os.path.join(DRIVE_UNIFEWS, 'Unifews', 'run_fb.py')):
    DRIVE_CODE_ROOT = os.path.join(DRIVE_UNIFEWS, 'Unifews')
    print(f'⚠️ Detected nested folder: {DRIVE_CODE_ROOT}')
else:
    raise FileNotFoundError(f'Cannot find run_fb.py in {DRIVE_UNIFEWS} or {DRIVE_UNIFEWS}/Unifews/')

# Only copy code & config (lightweight), symlink data/ from Drive (heavy)
LOCAL_UNIFEWS = '/content/Unifews'
if os.path.exists(LOCAL_UNIFEWS):
    !rm -rf /content/Unifews  # clean stale copy
    print('Cleaned stale copy.')

os.makedirs(LOCAL_UNIFEWS, exist_ok=True)
print('Copying code files only (skipping data/ and save/)...')
# Copy only code, config, and small files
!rsync -a --exclude='data/' --exclude='save/' --exclude='save_w1/' --exclude='raw/' --exclude='.git/' --exclude='__pycache__/' "{DRIVE_CODE_ROOT}/" /content/Unifews/
# Symlink heavy directories from Drive (no copy needed)
data_src = os.path.join(DRIVE_CODE_ROOT, 'data')
if not os.path.exists(data_src):
    # data might be in the outer folder
    data_src = os.path.join(DRIVE_UNIFEWS, 'data')
!ln -sf "{data_src}" /content/Unifews/data
# Create save dir locally
os.makedirs(f'{LOCAL_UNIFEWS}/save', exist_ok=True)
print('Done! (code copied, data/ symlinked from Drive)')

# Verify
assert os.path.exists(f'{LOCAL_UNIFEWS}/run_fb.py'), f'run_fb.py not found!'
assert os.path.exists(f'{LOCAL_UNIFEWS}/data'), f'data/ not found!'
assert os.path.exists(f'{LOCAL_UNIFEWS}/utils'), f'utils/ not found!'
print(f'✅ Unifews ready at {LOCAL_UNIFEWS}')
!ls {LOCAL_UNIFEWS}/

In [ ]:
# Install dependencies
!pip install -q ptflops dotmap powerlaw torch_geometric

# Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', None)
    if mem:
        print(f'GPU Memory: {mem / 1024**3:.1f} GB')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → GPU')

## 1. Download & Prepare Datasets

In [ ]:
import subprocess
import json
import csv
import pandas as pd
import numpy as np
import scipy.sparse as sp
from pathlib import Path

# Paths
UNIFEWS_DIR = Path('/content/Unifews')
CONFIG_DIR = UNIFEWS_DIR / 'config'
DATA_DIR = UNIFEWS_DIR / 'data'
SAVE_DIR = UNIFEWS_DIR / 'save'
SAVE_DIR.mkdir(exist_ok=True)

os.chdir(UNIFEWS_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Available datasets: {sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])}')
print(f'Available configs: {sorted([f.stem for f in CONFIG_DIR.glob("*.json")])}')

In [ ]:
"""Download heterophilic datasets using PyTorch Geometric."""
import torch

def download_and_convert_pyg_dataset(dataset_name, data_dir):
    from torch_geometric.datasets import WikipediaNetwork, WebKB, Planetoid
    
    save_path = data_dir / dataset_name
    if save_path.exists() and (save_path / 'adj.npz').exists():
        print(f'  {dataset_name}: already exists, skipping')
        return
    
    save_path.mkdir(parents=True, exist_ok=True)
    tmp_root = data_dir / 'raw'
    tmp_root.mkdir(exist_ok=True)
    
    if dataset_name in ['chameleon', 'squirrel']:
        dataset = WikipediaNetwork(root=str(tmp_root), name=dataset_name)
    elif dataset_name in ['cornell', 'texas', 'wisconsin']:
        dataset = WebKB(root=str(tmp_root), name=dataset_name)
    elif dataset_name in ['citeseer', 'pubmed']:
        dataset = Planetoid(root=str(tmp_root), name=dataset_name)
    else:
        raise ValueError(f'Unknown dataset: {dataset_name}')
    
    graph = dataset[0]
    n = graph.num_nodes
    
    row = graph.edge_index[0].numpy()
    col = graph.edge_index[1].numpy()
    row_full = np.concatenate([row, col])
    col_full = np.concatenate([col, row])
    data = np.ones(len(row_full), dtype=np.int8)
    adj = sp.csr_matrix((data, (row_full, col_full)), shape=(n, n))
    adj.setdiag(0)
    adj.eliminate_zeros()
    adj.data = np.ones(adj.nnz, dtype=np.int8)
    
    feats = graph.x.numpy()
    labels = graph.y.numpy().flatten()
    
    if hasattr(graph, 'train_mask') and graph.train_mask is not None:
        if graph.train_mask.dim() > 1:
            idx_train = torch.where(graph.train_mask[:, 0])[0].numpy()
            idx_val = torch.where(graph.val_mask[:, 0])[0].numpy()
            idx_test = torch.where(graph.test_mask[:, 0])[0].numpy()
        else:
            idx_train = torch.where(graph.train_mask)[0].numpy()
            idx_val = torch.where(graph.val_mask)[0].numpy()
            idx_test = torch.where(graph.test_mask)[0].numpy()
    else:
        perm = np.random.RandomState(42).permutation(n)
        n_train = int(0.5 * n)
        n_val = int(0.25 * n)
        idx_train = perm[:n_train]
        idx_val = perm[n_train:n_train+n_val]
        idx_test = perm[n_train+n_val:]
    
    sp.save_npz(str(save_path / 'adj.npz'), adj.tocsc())
    np.save(str(save_path / 'feats.npy'), feats)
    np.savez(str(save_path / 'labels.npz'),
             labels=labels, idx_train=idx_train, idx_val=idx_val, idx_test=idx_test)
    degree = np.array(adj.sum(axis=1)).flatten()
    sp.save_npz(str(save_path / 'degree.npz'), sp.csr_matrix(degree.reshape(-1, 1)))
    
    rows, cols = adj.nonzero()
    homophily = (labels[rows] == labels[cols]).sum() / len(rows) if len(rows) > 0 else 0
    print(f'  {dataset_name}: n={n}, m={adj.nnz}, f={feats.shape[1]}, '
          f'classes={len(np.unique(labels))}, h={homophily:.3f}')

print('=== Downloading missing datasets ===')
for ds in ['chameleon', 'cornell', 'wisconsin', 'texas', 'citeseer', 'pubmed']:
    try:
        download_and_convert_pyg_dataset(ds, DATA_DIR)
    except Exception as e:
        print(f'  {ds}: FAILED - {e}')

print('\n=== All datasets ===')
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir() and (d / 'adj.npz').exists():
        adj = sp.load_npz(str(d / 'adj.npz'))
        print(f'  {d.name}: n={adj.shape[0]}, m={adj.nnz}')

## 2. Helper Functions

In [ ]:
import re
import time

def run_experiment(config, algo='gcn_thr', seed=42, thr_a=0.5, thr_w=0.5,
                   fa_alpha=1.0, device=0, extra_args=None):
    """Run a single experiment via run_fb.py and return result dict."""
    cmd = [
        'python', 'run_fb.py',
        '-f', str(seed),
        '-c', config,
        '-m', algo,
        '-a', str(thr_a),
        '-w', str(thr_w),
        '-v', str(device),
        '--fa_alpha', str(fa_alpha),
    ]
    if extra_args:
        cmd.extend(extra_args)
    
    t0 = time.time()
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        cwd=str(UNIFEWS_DIR), timeout=600
    )
    elapsed = time.time() - t0
    
    if result.returncode != 0:
        print(f'  ERROR ({elapsed:.0f}s): {result.stderr[-300:]}')
        return {'config': config, 'algo': algo, 'seed': seed,
                'thr_a': thr_a, 'thr_w': thr_w, 'fa_alpha': fa_alpha,
                'acc': None, 'error': result.stderr[-500:]}
    
    output = result.stdout + result.stderr
    acc = numel_a = numel_w = time_train = time_test = macs_train = macs_test = None
    
    for line in output.split('\n'):
        if '[test]' in line.lower() and 'best acc' in line.lower():
            m = re.search(r'best acc:\s*([0-9.]+)', line, re.IGNORECASE)
            if m: acc = float(m.group(1))
        if '[test]' in line.lower() and 'num adj' in line.lower():
            m_na = re.search(r'Num adj:\s*([0-9.]+)', line)
            m_nw = re.search(r'Num weight:\s*([0-9.]+)', line)
            m_tt = re.search(r'time:\s*([0-9.]+)', line)
            m_mc = re.search(r'MACs:\s*([0-9.]+)', line)
            if m_na: numel_a = float(m_na.group(1))
            if m_nw: numel_w = float(m_nw.group(1))
            if m_tt: time_test = float(m_tt.group(1))
            if m_mc: macs_test = float(m_mc.group(1))
        if '[train]' in line.lower() and 'time:' in line.lower():
            m_tr = re.search(r'time:\s*([0-9.]+)', line)
            m_mc = re.search(r'MACs:\s*([0-9.]+)', line)
            if m_tr: time_train = float(m_tr.group(1))
            if m_mc: macs_train = float(m_mc.group(1))
        # Fallback: CSV line
        if acc is None and ',' in line:
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 14:
                try:
                    maybe_acc = float(parts[5])
                    if 0 < maybe_acc < 1:
                        acc = maybe_acc
                        numel_a = float(parts[12])
                        numel_w = float(parts[13])
                except (ValueError, IndexError):
                    pass
    
    return {
        'config': config, 'algo': algo, 'seed': seed,
        'thr_a': thr_a, 'thr_w': thr_w, 'fa_alpha': fa_alpha,
        'acc': acc, 'numel_a': numel_a, 'numel_w': numel_w,
        'time_train': time_train, 'time_test': time_test,
        'macs_train': macs_train, 'macs_test': macs_test,
        'wall_time': elapsed,
    }


def run_experiment_batch(experiments, desc=''):
    """Run a batch of experiments with progress tracking."""
    results = []
    total = len(experiments)
    t_start = time.time()
    print(f'\n{"="*60}')
    print(f'Running {total} experiments: {desc}')
    print(f'{"="*60}')
    
    for i, exp in enumerate(experiments):
        print(f'\n[{i+1}/{total}] {exp.get("config","?")} | '
              f'fa_alpha={exp.get("fa_alpha","?")} | thr_a={exp.get("thr_a","?")}')
        try:
            r = run_experiment(**exp)
            results.append(r)
            if r and r.get('acc'):
                print(f'  → acc={r["acc"]:.4f} ({r["wall_time"]:.0f}s)')
            else:
                print(f'  → FAILED: {r.get("error", "no accuracy parsed")[:100]}')
        except subprocess.TimeoutExpired:
            print(f'  → TIMEOUT (>600s)')
            results.append({**exp, 'acc': None, 'error': 'timeout'})
        except Exception as e:
            print(f'  → EXCEPTION: {e}')
            results.append({**exp, 'acc': None, 'error': str(e)})
    
    elapsed = time.time() - t_start
    n_ok = sum(1 for r in results if r.get('acc') is not None)
    print(f'\n{"="*60}')
    print(f'Done: {n_ok}/{total} succeeded in {elapsed:.0f}s ({elapsed/60:.1f} min)')
    print(f'{"="*60}')
    return results

print('Helper functions defined. ✅')

## 3. Quick Test (single run)

In [ ]:
# Quick test: single Cora run to verify everything works
print('Testing single experiment on Cora...')
test_result = run_experiment(config='cora', algo='gcn_thr', seed=42,
                             thr_a=0.7, thr_w=0.5, fa_alpha=0.5, device=0)
if test_result and test_result.get('acc'):
    print(f'\n✅ Test passed! acc={test_result["acc"]:.4f} in {test_result["wall_time"]:.1f}s')
else:
    print(f'\n❌ Test failed!')
    print(test_result.get('error', 'Unknown error')[:500])

## 4. Experiment 1: Main Comparison (Table 3)

In [ ]:
DATASETS = ['cora', 'cs', 'computers', 'chameleon', 'cornell', 'wisconsin']
FA_MODES = {
    'UNIFEWS':       1.0,
    'FA-UNI_static': 0.5,
    'FA-UNI_adapt': -1.0,
}
THR_A = 0.7
THR_W = 0.5
SEED = 42

experiments_main = []
for dataset in DATASETS:
    for mode_name, fa_alpha in FA_MODES.items():
        experiments_main.append({
            'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
            'thr_a': THR_A, 'thr_w': THR_W, 'fa_alpha': fa_alpha,
        })

results_main = run_experiment_batch(experiments_main, desc='Main Comparison (Table 3)')

# Summary table
print('\n' + '='*70)
print(f'{"Dataset":>12s} | {"UNIFEWS":>10s} | {"FA-static":>10s} | {"FA-adapt":>10s} | {"Δ static":>10s}')
print('-'*70)
for ds in DATASETS:
    accs = {}
    for r in results_main:
        if r['config'] == ds and r.get('acc'):
            accs[r['fa_alpha']] = r['acc'] * 100
    u = accs.get(1.0, 0)
    s = accs.get(0.5, 0)
    a = accs.get(-1.0, 0)
    delta = s - u if u and s else 0
    print(f'{ds:>12s} | {u:>9.2f}% | {s:>9.2f}% | {a:>9.2f}% | {delta:>+9.2f}%')

## 5. Experiment 2: MLP Baselines

In [ ]:
experiments_mlp = []
for dataset in DATASETS:
    experiments_mlp.append({
        'config': dataset, 'algo': 'mlp', 'seed': SEED,
        'thr_a': 0.0, 'thr_w': 0.0, 'fa_alpha': 1.0,
    })

results_mlp = run_experiment_batch(experiments_mlp, desc='MLP Baselines')

## 6. Experiment 3: Dense GCN Baselines

In [ ]:
experiments_dense = []
for dataset in DATASETS:
    experiments_dense.append({
        'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
        'thr_a': 0.0, 'thr_w': 0.0, 'fa_alpha': 1.0,
    })

results_dense = run_experiment_batch(experiments_dense, desc='Dense GCN Baselines')

## 7. Experiment 4: Ablation Study (Table 6)

In [ ]:
ABLATION_DATASETS = ['cora', 'chameleon']
ALPHA_SWEEP = [0.1, 0.3, 0.5, 0.7, 0.9]

experiments_ablation = []
for dataset in ABLATION_DATASETS:
    # Full adaptive
    experiments_ablation.append({
        'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
        'thr_a': THR_A, 'thr_w': THR_W, 'fa_alpha': -1.0,
    })
    # UNIFEWS baseline
    experiments_ablation.append({
        'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
        'thr_a': THR_A, 'thr_w': THR_W, 'fa_alpha': 1.0,
    })
    # Alpha sweep
    for alpha in ALPHA_SWEEP:
        experiments_ablation.append({
            'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
            'thr_a': THR_A, 'thr_w': THR_W, 'fa_alpha': alpha,
        })

results_ablation = run_experiment_batch(experiments_ablation, desc='Ablation Study')

# Summary
print('\n' + '='*50)
print(f'{"Config":>30s} | {"Cora":>8s} | {"Chameleon":>10s}')
print('-'*50)
for r in results_ablation:
    if r['config'] == 'cora' and r.get('acc'):
        label = f'fa_alpha={r["fa_alpha"]}'
        # Find matching chameleon
        ch = [x for x in results_ablation if x['config']=='chameleon' and x['fa_alpha']==r['fa_alpha']]
        ch_acc = ch[0]['acc']*100 if ch and ch[0].get('acc') else 0
        print(f'{label:>30s} | {r["acc"]*100:>7.2f}% | {ch_acc:>9.2f}%')

## 8. Experiment 5: Sparsity Sweep (Table 7)

In [ ]:
THR_A_VALUES = [0.0, 0.2, 0.4, 0.8, 1.0, 1.5, 2.0, 3.0]
SWEEP_DATASETS = ['cora', 'computers', 'cs']

experiments_sweep = []
for dataset in SWEEP_DATASETS:
    for thr_a in THR_A_VALUES:
        experiments_sweep.append({
            'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
            'thr_a': thr_a, 'thr_w': 0.0, 'fa_alpha': 1.0,
        })
        experiments_sweep.append({
            'config': dataset, 'algo': 'gcn_thr', 'seed': SEED,
            'thr_a': thr_a, 'thr_w': 0.0, 'fa_alpha': 0.5,
        })

results_sweep = run_experiment_batch(experiments_sweep, desc='Sparsity Sweep')

## 9. Compute Edge Homophily

In [ ]:
def compute_edge_homophily(dataset_name):
    data_path = DATA_DIR / dataset_name
    adj = sp.load_npz(str(data_path / 'adj.npz'))
    labels = np.load(str(data_path / 'labels.npz'), allow_pickle=True)['labels']
    rows, cols = adj.nonzero()
    if len(rows) == 0: return 0.0
    return (labels[rows] == labels[cols]).sum() / len(rows)

print('=== Original Edge Homophily ===')
for ds in DATASETS:
    try:
        h = compute_edge_homophily(ds)
        print(f'  {ds:>12s}: H = {h:.4f}')
    except Exception as e:
        print(f'  {ds:>12s}: ERROR - {e}')

## 10. Aggregate Results & Generate LaTeX Tables

In [ ]:
def results_to_df(results, label=''):
    rows = []
    for r in results:
        if r is None: continue
        rows.append({
            'config': r.get('config',''), 'algo': r.get('algo',''),
            'fa_alpha': r.get('fa_alpha',''), 'thr_a': r.get('thr_a',''),
            'thr_w': r.get('thr_w',''), 'acc': r.get('acc'),
            'numel_a': r.get('numel_a'), 'numel_w': r.get('numel_w'),
            'time_train': r.get('time_train'), 'time_test': r.get('time_test'),
            'macs_train': r.get('macs_train'), 'macs_test': r.get('macs_test'),
            'wall_time': r.get('wall_time'), 'label': label,
        })
    return pd.DataFrame(rows)

# Combine all
all_dfs = []
for name, var in [('main','results_main'), ('mlp','results_mlp'),
                  ('dense','results_dense'), ('ablation','results_ablation'),
                  ('sweep','results_sweep')]:
    data = globals().get(var, [])
    if data:
        all_dfs.append(results_to_df(data, label=name))

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f'Total results: {len(df_all)}')
    
    # Save to CSV
    output_path = SAVE_DIR / 'all_fa_unifews_results.csv'
    df_all.to_csv(output_path, index=False)
    print(f'Saved to: {output_path}')
    
    # Also save to Drive for persistence
    drive_save = Path(DRIVE_UNIFEWS) / 'save'
    drive_save.mkdir(exist_ok=True)
    df_all.to_csv(drive_save / 'all_fa_unifews_results.csv', index=False)
    print(f'Saved to Drive: {drive_save / "all_fa_unifews_results.csv"}')
    
    # Show results with acc
    df_valid = df_all[df_all['acc'].notna()]
    print(f'\nValid results: {len(df_valid)}/{len(df_all)}')
    display(df_valid[['config','algo','fa_alpha','thr_a','acc','wall_time','label']].to_string())
else:
    print('No results collected yet.')

In [ ]:
# ============================================================
# Generate LaTeX table for paper (Table 3: Main Results)
# ============================================================
def generate_main_table_latex(df):
    datasets = ['cora', 'cs', 'computers', 'chameleon', 'cornell', 'wisconsin']
    methods = {
        'Dense GCN':                {'label': 'dense', 'fa_alpha': 1.0, 'thr_a': 0.0},
        'MLP':                      {'label': 'mlp',   'fa_alpha': 1.0, 'thr_a': 0.0},
        'UNIFEWS':                  {'label': 'main',  'fa_alpha': 1.0, 'thr_a': 0.7},
        r'FA-UNI$_\text{static}$': {'label': 'main',  'fa_alpha': 0.5, 'thr_a': 0.7},
        r'FA-UNI$_\text{adapt}$':  {'label': 'main',  'fa_alpha': -1.0,'thr_a': 0.7},
    }
    
    print('% Auto-generated LaTeX table')
    print(r'\begin{tabular}{lcccccc}')
    print(r'\toprule')
    print(r' & \textbf{Cora} & \textbf{CS} & \textbf{Comp.} & '
          r'\textbf{Cham.} & \textbf{Cornell} & \textbf{Wisc.} \\')
    print(r'\midrule')
    
    for method_name, criteria in methods.items():
        row = [method_name]
        for ds in datasets:
            mask = (df['config'] == ds) & (df['label'] == criteria['label'])
            if criteria['label'] == 'main':
                mask = mask & (df['fa_alpha'] == criteria['fa_alpha'])
            subset = df[mask]
            if len(subset) > 0 and subset.iloc[0]['acc'] is not None:
                acc = subset.iloc[0]['acc'] * 100 if subset.iloc[0]['acc'] < 1 else subset.iloc[0]['acc']
                row.append(f'{acc:.2f}')
            else:
                row.append('--')
        print(' & '.join(row) + r' \\')
    
    print(r'\bottomrule')
    print(r'\end{tabular}')

if 'df_all' in dir() and len(df_all) > 0:
    generate_main_table_latex(df_all)
else:
    print('Run experiments first.')

## 11. Multi-Seed (Final Paper Numbers)

⚠️ **Only run this when single-seed results look good.**  
3 methods × 6 datasets × 10 seeds = 180 runs (~30 min on T4)

In [ ]:
# Uncomment to run multi-seed
SEEDS = [11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

experiments_multiseed = []
for dataset in DATASETS:
    for mode_name, fa_alpha in FA_MODES.items():
        for seed in SEEDS:
            experiments_multiseed.append({
                'config': dataset, 'algo': 'gcn_thr', 'seed': seed,
                'thr_a': THR_A, 'thr_w': THR_W, 'fa_alpha': fa_alpha,
            })

print(f'Total: {len(experiments_multiseed)} runs (~30 min on T4 GPU)')
print('Uncomment next line to execute:')
# results_multiseed = run_experiment_batch(experiments_multiseed, 'Multi-seed Main')

In [ ]:
# After running multi-seed, compute mean ± std
if 'results_multiseed' in dir():
    df_ms = results_to_df(results_multiseed, label='multiseed')
    df_ms_valid = df_ms[df_ms['acc'].notna()].copy()
    df_ms_valid['acc_pct'] = df_ms_valid['acc'] * 100
    
    summary = df_ms_valid.groupby(['config', 'fa_alpha'])['acc_pct'].agg(['mean','std','count'])
    summary = summary.reset_index()
    summary['result'] = summary.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}", axis=1)
    
    print('\n=== Multi-seed Results (mean ± std) ===')
    for ds in DATASETS:
        sub = summary[summary['config'] == ds]
        print(f'\n{ds}:')
        for _, row in sub.iterrows():
            print(f'  fa_alpha={row["fa_alpha"]:5.1f}: {row["result"]} (n={row["count"]:.0f})')
    
    # Save
    df_ms.to_csv(SAVE_DIR / 'multiseed_results.csv', index=False)
    drive_save = Path(DRIVE_UNIFEWS) / 'save'
    df_ms.to_csv(drive_save / 'multiseed_results.csv', index=False)
    print(f'\nSaved to Drive.')
else:
    print('Run multi-seed experiments first (cell above).')

## 12. Save Everything Back to Drive

In [ ]:
# Copy save/ and any new data/ back to Google Drive
import shutil

drive_save = Path(DRIVE_UNIFEWS) / 'save'
drive_save.mkdir(exist_ok=True)

# Copy all save folders
for item in (UNIFEWS_DIR / 'save').iterdir():
    dst = drive_save / item.name
    if item.is_dir():
        if dst.exists():
            shutil.rmtree(str(dst))
        shutil.copytree(str(item), str(dst))
    else:
        shutil.copy2(str(item), str(dst))

print(f'✅ Results synced to Google Drive: {drive_save}')
print('\nFiles saved:')
for f in sorted(drive_save.rglob('*.csv')):
    print(f'  {f.relative_to(drive_save)}')